# Phi-4-reasoning-vision-15B – Task 1 K-fold Evaluation

Evaluates direct sentiment classification results produced by Microsoft Phi-4-reasoning-vision-15B  
(Task 1: direct image classification, no description mediation).  

Run `phi4_task1_classify.py` (or `run-phi4-task1.sh`) first to generate the responses in `data/phi4-only/`.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, accuracy_score
from scipy import stats

## Configurations

All four Task 1 variants are evaluated automatically.  
Sigma (σ) refers to the label-noise threshold; P3/P5 is the number of sentiment classes.

In [2]:
BASE     = "/mnt/raid5/neemias/PerceptSent-LLM-approach"
GT_DIR   = f"{BASE}/data/gpt4-openai-classify"
PHI4_DIR = f"{BASE}/data/phi4-only"

SENT_MAP_P5 = {"Positive": 4, "Slightlypositive": 3, "Neutral": 2, "Slightlynegative": 1, "Negative": 0}
SENT_MAP_P3 = {"Positive": 2, "Neutral": 0, "Negative": 1}

CONFIGS = {
    "P3 + alpha3 (σ3)": {
        "gt":   f"{GT_DIR}/percept_dataset_alpha3_p3.csv",
        "phi4": f"{PHI4_DIR}/alpha3p3.csv",
    },
    "P5 + alpha3 (σ3)": {
        "gt":   f"{GT_DIR}/percept_dataset_alpha3_p5.csv",
        "phi4": f"{PHI4_DIR}/alpha3p5.csv",
    },
    "P3 + alpha5 (σ5)": {
        "gt":   f"{GT_DIR}/percept_dataset_alpha5_p3.csv",
        "phi4": f"{PHI4_DIR}/alpha5p3.csv",
    },
    "P5 + alpha5 (σ5)": {
        "gt":   f"{GT_DIR}/percept_dataset_alpha5_p5.csv",
        "phi4": f"{PHI4_DIR}/alpha5p5.csv",
    },
}
print("Configurations:", list(CONFIGS.keys()))

Configurations: ['P3 + alpha3 (σ3)', 'P5 + alpha3 (σ3)', 'P3 + alpha5 (σ5)', 'P5 + alpha5 (σ5)']


In [3]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
all_results = {}

for cfg_name, cfg in CONFIGS.items():
    df_gt   = pd.read_csv(cfg["gt"])
    df_resp = pd.read_csv(cfg["phi4"])
    df_resp["sentiment"] = df_resp["sentiment"].astype(str).str.replace(".", "", regex=False).str.strip()

    common_ids = set(df_gt["id"]) & set(df_resp["id"])
    df_gt   = df_gt[df_gt["id"].isin(common_ids)].sort_values("id").reset_index(drop=True)
    df_resp = df_resp[df_resp["id"].isin(common_ids)].sort_values("id").reset_index(drop=True)

    sent_dic = SENT_MAP_P5 if df_gt["sentiment"].nunique() == 5 else SENT_MAP_P3

    rows = []
    for fold, (_, val_idx) in enumerate(kfold.split(df_gt)):
        val_gt = df_gt.iloc[val_idx]
        target = val_gt["sentiment"].tolist()
        ids    = val_gt["id"].tolist()

        pred = []
        for img_id in ids:
            s   = df_resp[df_resp["id"] == img_id]["sentiment"].iloc[0]
            key = s.replace(" ", "")
            if key not in sent_dic:
                key = next((k for k in sent_dic if k.lower() == key.lower()), list(sent_dic.keys())[0])
            pred.append(sent_dic[key])

        rows.append({
            "fold":     fold,
            "accuracy": accuracy_score(target, pred),
            "f1_score": f1_score(target, pred, average="weighted"),
        })

    df_fold  = pd.DataFrame(rows)
    f1s      = df_fold["f1_score"].tolist()
    accs     = df_fold["accuracy"].tolist()
    mean_f1  = np.mean(f1s)
    mean_acc = np.mean(accs)
    ci       = stats.t.interval(0.95, len(f1s) - 1, loc=mean_f1, scale=stats.sem(f1s))
    all_results[cfg_name] = dict(mean_f1=mean_f1, mean_acc=mean_acc, ci=ci, df=df_fold)

    print(f"\n{'='*55}")
    print(f"  Task 1  |  {cfg_name}")
    print(f"{'='*55}")
    display(df_fold)
    print(f"Max  F1  : {max(f1s):.4f}")
    print(f"Mean F1  : {mean_f1:.4f}")
    print(f"Mean Acc : {mean_acc:.4f}")
    print(f"95% CI   : [{ci[0]:.4f}, {ci[1]:.4f}]  ±{abs(ci[1] - mean_f1):.4f}")


  Task 1  |  P3 + alpha3 (σ3)


,fold,accuracy,f1_score
0,0,0.589944,0.628526
1,1,0.605587,0.643314
2,2,0.550336,0.591128
3,3,0.549217,0.598483
4,4,0.557047,0.596323


Max  F1  : 0.6433
Mean F1  : 0.6116
Mean Acc : 0.5704
95% CI   : [0.5830, 0.6401]  ±0.0286



  Task 1  |  P5 + alpha3 (σ3)


,fold,accuracy,f1_score
0,0,0.539548,0.451505
1,1,0.502122,0.406655
2,2,0.506365,0.409844
3,3,0.478076,0.386940
4,4,0.550212,0.454457


...


  Task 1  |  P3 + alpha5 (σ5)


,fold,accuracy,f1_score
0,0,0.729730,0.820642
1,1,0.735736,0.812742
2,2,0.765060,0.831042
3,3,0.765060,0.835031
4,4,0.740964,0.805530


Max  F1  : 0.8350
Mean F1  : 0.8210
Mean Acc : 0.7473
95% CI   : [0.8057, 0.8363]  ±0.0153

  Task 1  |  P5 + alpha5 (σ5)


,fold,accuracy,f1_score
0,0,0.852273,0.797417
1,1,0.772727,0.716730
2,2,0.781609,0.735140
3,3,0.885057,0.846885
4,4,0.839080,0.808071


Max  F1  : 0.8469
Mean F1  : 0.7808
Mean Acc : 0.8261
95% CI   : [0.7141, 0.8476]  ±0.0668


P5 + sigma3 = 0.42 ± 0.0286
P3 + sigma3 = 0.61 ± 0.0228
P5 + sigma5 = 0.78 ± 0.0668
P3 + sigma5 = 0.82 ± 0.0153

## Task 2a – ModernBERT (no fine-tuning)

Results from `experiments-not-finetuning/phi4-modernbert-experiment-p3-alpha3`.

In [4]:
EXP_DIR_2A = f"{BASE}/experiments-not-finetuning"

CONFIGS_2A = {
    "P3 + alpha3 (σ3)": f"{EXP_DIR_2A}/phi4-modernbert-experiment-p3-alpha3/logs/test_logs.csv",
    "P5 + alpha3 (σ3)": f"{EXP_DIR_2A}/phi4-modernbert-experiment-p5-alpha3/logs/test_logs.csv",
    "P3 + alpha5 (σ5)": f"{EXP_DIR_2A}/phi4-modernbert-experiment-p3-alpha5/logs/test_logs.csv",
    "P5 + alpha5 (σ5)": f"{EXP_DIR_2A}/phi4-modernbert-experiment-p5-alpha5/logs/test_logs.csv",
}

for cfg_name, log_path in CONFIGS_2A.items():
    df = pd.read_csv(log_path)
    f1s  = df["f1_score"].tolist()
    accs = df["accuracy"].tolist()
    mean_f1  = np.mean(f1s)
    mean_acc = np.mean(accs)
    ci = stats.t.interval(0.95, len(f1s) - 1, loc=mean_f1, scale=stats.sem(f1s))

    print(f"\n{'='*55}")
    print(f"  Task 2a  |  {cfg_name}")
    print(f"{'='*55}")
    display(df)
    print(f"Max  F1  : {max(f1s):.4f}")
    print(f"Mean F1  : {mean_f1:.4f}")
    print(f"Mean Acc : {mean_acc:.4f}")
    print(f"95% CI   : [{ci[0]:.4f}, {ci[1]:.4f}]  ±{abs(ci[1] - mean_f1):.4f}")


  Task 2a  |  P3 + alpha3 (σ3)


,kfold,accuracy,f1_score,time
0,1,0.679330,0.691435,20146
1,2,0.437989,0.266809,14801
2,3,0.646532,0.663486,20179
3,4,0.705817,0.713476,20265
4,5,0.365772,0.195917,10330


Max  F1  : 0.7135
Mean F1  : 0.5062
Mean Acc : 0.5671
95% CI   : [0.1924, 0.8201]  ±0.3139



  Task 2a  |  P5 + alpha3 (σ3)


,kfold,accuracy,f1_score,time
0,1,0.478814,0.460275,15581
1,2,0.420085,0.413391,14464
2,3,0.268741,0.114447,7688
3,4,0.169731,0.049317,7398
4,5,0.230552,0.086391,7682


Max  F1  : 0.4603
Mean F1  : 0.2248
Mean Acc : 0.3136
95% CI   : [-0.0182, 0.4677]  ±0.2430

  Task 2a  |  P3 + alpha5 (σ5)


,kfold,accuracy,f1_score,time
0,1,0.867868,0.882444,6797
1,2,0.843844,0.856581,6804
2,3,0.876877,0.862357,6801
3,4,0.846847,0.870115,6787
4,5,0.903614,0.920794,6786


Max  F1  : 0.9208
Mean F1  : 0.8785
Mean Acc : 0.8678
95% CI   : [0.8467, 0.9102]  ±0.0318

  Task 2a  |  P5 + alpha5 (σ5)


,kfold,accuracy,f1_score,time
0,1,0.090909,0.015152,991
1,2,0.193182,0.062554,1076
2,3,0.125000,0.027778,1005
3,4,0.579545,0.611299,1791
4,5,0.022989,0.001033,946


Max  F1  : 0.6113
Mean F1  : 0.1436
Mean Acc : 0.2023
95% CI   : [-0.1823, 0.4695]  ±0.3259


## Task 2b – ModernBERT (fine-tuned)

Results from `experiments-finetuning/phi4-modernbert-experiment-p3-alpha3`.

In [5]:
EXP_DIR_2B = f"{BASE}/experiments-finetuning"

CONFIGS_2B = {
    "P3 + alpha3 (σ3)": f"{EXP_DIR_2B}/phi4-modernbert-experiment-p3-alpha3/logs/test_logs.csv",
    "P5 + alpha3 (σ3)": f"{EXP_DIR_2B}/phi4-modernbert-experiment-p5-alpha3/logs/test_logs.csv",
    "P3 + alpha5 (σ5)": f"{EXP_DIR_2B}/phi4-modernbert-experiment-p3-alpha5/logs/test_logs.csv",
    "P5 + alpha5 (σ5)": f"{EXP_DIR_2B}/phi4-modernbert-experiment-p5-alpha5/logs/test_logs.csv",
}

for cfg_name, log_path in CONFIGS_2B.items():
    df = pd.read_csv(log_path)
    f1s  = df["f1_score"].tolist()
    accs = df["accuracy"].tolist()
    mean_f1  = np.mean(f1s)
    mean_acc = np.mean(accs)
    ci = stats.t.interval(0.95, len(f1s) - 1, loc=mean_f1, scale=stats.sem(f1s))

    print(f"\n{'='*55}")
    print(f"  Task 2b  |  {cfg_name}")
    print(f"{'='*55}")
    display(df)
    print(f"Max  F1  : {max(f1s):.4f}")
    print(f"Mean F1  : {mean_f1:.4f}")
    print(f"Mean Acc : {mean_acc:.4f}")
    print(f"95% CI   : [{ci[0]:.4f}, {ci[1]:.4f}]  ±{abs(ci[1] - mean_f1):.4f}")


  Task 2b  |  P3 + alpha3 (σ3)


,kfold,accuracy,f1_score,time
0,1,0.726257,0.719969,6128
1,2,0.717318,0.717750,9401
2,3,0.724832,0.713590,7086
3,4,0.741611,0.737290,7568
4,5,0.767338,0.765795,7645


Max  F1  : 0.7658
Mean F1  : 0.7309
Mean Acc : 0.7355
95% CI   : [0.7042, 0.7576]  ±0.0267



  Task 2b  |  P5 + alpha3 (σ3)


,kfold,accuracy,f1_score,time
0,1,0.528249,0.525723,5608
1,2,0.513437,0.517469,10444
2,3,0.550212,0.552241,7091
3,4,0.553041,0.557319,8204
4,5,0.506365,0.505006,7095


...


  Task 2b  |  P3 + alpha5 (σ5)


,kfold,accuracy,f1_score,time
0,1,0.936937,0.938754,3343
1,2,0.942943,0.942746,3166
2,3,0.912913,0.909000,4218
3,4,0.927928,0.924753,3693
4,5,0.945783,0.950884,2991


Max  F1  : 0.9509
Mean F1  : 0.9332
Mean Acc : 0.9333
95% CI   : [0.9127, 0.9537]  ±0.0205

  Task 2b  |  P5 + alpha5 (σ5)


,kfold,accuracy,f1_score,time
0,1,0.784091,0.742803,891
1,2,0.818182,0.826360,650
2,3,0.761364,0.741616,1806
3,4,0.806818,0.824172,742
4,5,0.816092,0.817063,883


Max  F1  : 0.8264
Mean F1  : 0.7904
Mean Acc : 0.7973
95% CI   : [0.7356, 0.8452]  ±0.0548
